# process one subject's fMRI data 

clip study test nii.gz files so they're the same length (based on events.tsv) such that we can feed into tc.timecorr to get correlation matrices for forget vs remember conditions 

### note: 
nii.gz in TRs not seconds (conversion: 1 TR = 2 sec)


---
**events.tsv**
- onset: this tells us when the event starts (in seconds) relative to the start of the neuroimaging file
- duration: this tells us how long the given event lasts (should always be 3 seconds)
- list: are we studying list A or list B?
- word: what word is the participant looking at now?
- memory cue: will the participant be told to remember or forget the list A words on this run?  (not relevant for list B words)

In [154]:
import pandas as pd
import numpy as np
import nibabel as nib
from bids import BIDSLayout

In [155]:
# setup 
data_dir = "../../data-cdl"
layout = BIDSLayout(data_dir, derivatives=True)
subs = layout.get_subjects() 
subj_niftis = layout.get(subject=sub_ID, return_type='file', task="StudyTest", extension='nii.gz')

c:\Users\mgnli\anaconda3\envs\timecorr\lib\site-packages\bids\layout\validation.py:131: UserWarning: Derivative indexing was requested, but no valid derivative datasets were found in the specified locations ([WindowsPath('c:/Users/mgnli/CDL/directed-forgetting-network-dynamics/code/../../data-cdl/derivatives')]). Note that all BIDS-Derivatives datasets must meet all the requirements for BIDS-Raw datasets (a common problem is to fail to include a 'dataset_description.json' file in derivatives datasets).
Example contents of 'dataset_description.json':
{"Name": "Example dataset", "BIDSVersion": "1.0.2", "GeneratedBy": [{"Name": "Example pipeline"}]}
  warnings.warn("Derivative indexing was requested, but no valid "


In [156]:
"""
processes one events.tsv file from one run from one subject and returns tsv as a pandas dataframe and forget/remember cue 

input: 
    run_ID - str 
    sub_ID - str (str is two chars, consisting of ints)
output: 
    sub_run_df - pandas df
    cue - str (forget or remember)
"""
def process_tsv(run_ID, sub_ID): 
    sub_run_df = pd.read_csv(f'{data_dir}/sub-{sub_ID}/func/sub-{sub_ID}_task-StudyTest_run-{run_ID}_events.tsv', sep='\t')

    cue = sub_run_df["memory_cue"][0]
    # check cue doesn't change for that run 
    for e in sub_run_df["memory_cue"]: 
        # print(e)
        if not ((e == cue) or (pd.isna(e))):
            cue = None 
            print("ERROR in events.tsv for sub", sub_ID, " run",run_ID)
    
    return sub_run_df, cue

In [157]:
"""
processes all events.tsv files from one subject and returns pandas dataframe of tsvs and forget/remember cues 

input: 
    sub_ID - str (str is two chars, consisting of ints)
output: 
    sub_dfs - array of pandas dataframes 
    sub_cues (forget/remember) -  array of str
    
    run # = index + 1 (e.g., run 1 = index 0)
"""
def process_one_sub_tsvs(sub_ID): 
    sub_dfs = []
    sub_cues = []
    
    for run in range(1, len(subj_niftis)+1):
        run_ID = f"{run:02}"

        sub_run_df, cue = process_tsv(run_ID, sub_ID)
       
        sub_dfs.append(sub_run_df)
        sub_cues.append(cue)
    return sub_dfs, sub_cues

In [163]:
"""
processes all events.tsv files from all subjects and returns nested array of subjects and their pandas dataframes and forget/remember cues 

input: 
    subs - array of str 
output: 
    dfs - nested array; outer array (index i = sub i-1) & inner arrays (sub i-1 = array of pandas dataframes where index e = run e - 1)
        since subjects and runs are indexed at 1 
    cues - nested array similar to dfs above
"""
def process_all_subs_tsvs(subs): 
    dfs = []
    cues = []

    for sub_ID in subs: 
        sub_dfs, sub_cues = process_one_sub_tsvs(sub_ID)
        
        dfs.append(sub_dfs)
        cues.append(sub_cues)

    return dfs, cues

In [172]:
def main(): 
    dfs, cues = process_all_subs_tsvs(subs)


    # TODO: check process_all_subs_tsvs() works
    # print(dfs)

    # count1 = 0 
    # count2 = 0
    # for e in dfs: 
    #     count1 += 1
    #     for i in dfs[e]: 
    #         count2 += 1

    # print(count1, count2)

main()

ERROR in events.tsv for sub 06  run 07


In [173]:
## TODO: NIFTI 
# each index is one subj. within each subj = 8 subj runs
# [[[S1R1],[S1R2],...,[S1R8]], [S2R1],[S2R2],...,[S2R8]], ..., [S24R1],[S24R2],...,[S24R8]]]
# niftis = [] 

# load data 
"""subj_niftis = layout.get(subject=sub_ID, return_type='file', task="StudyTest", extension='nii.gz')
print(subj_niftis)
print(len(subj_niftis))"""

# data = nib.load()
# niftis.append(data)

# find # TRs 
# convert to seconds 

# find run with latest onset for first stimulus 
# pad other runs with zeros so that first stimulus onset time matches 
# check intervals between stimulus is the same such that the other stimuli also match up  
    


'subj_niftis = layout.get(subject=sub_ID, return_type=\'file\', task="StudyTest", extension=\'nii.gz\')\nprint(subj_niftis)\nprint(len(subj_niftis))'